In [3]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph,END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document


ModuleNotFoundError: No module named 'langchain_openai'

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [4]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-4", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


TypeError: str expected, not NoneType

In [ ]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [ ]:
## Sample Documents and Vector Store Setup
sample_texts = [
    "LangGraph is a powerful framework for building language model applications.",
    "Retrieval-Augmented Generation (RAG) combines retrieval of documents with generation capabilities of LLMs.",
    "FAISS is a library for efficient similarity search and clustering of dense vectors.",
]
documents = [Document(page_content=text) for text in sample_texts]

vector_store = FAISS.from_documents(documents, embeddings)  
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})



In [ ]:
def decide_retrieval(state: AgentState) -> str:
    """
    Decide whether to retrieve documents based on the current state.
    """
    question = state["question"]
    reterival_keywords = ["who", "what", "when", "where", "why", "how", "explain", "define", "details about"]
    needs_retrieval = any(keyword in question.lower() for keyword in reterival_keywords)
    return {**state, "needs_retrieval": needs_retrieval}

In [ ]:
def retrieve_documents(state: AgentState) -> AgentState:
    """
    Retrieve relevant documents based on the question in the state.
    """
    question = state["question"]
    documents = retriever.invoke(question)
    return {**state, "documents": documents}

In [ ]:
def generate_answer(state: AgentState) -> AgentState:
    """
    Generate an answer based on the question and retrieved documents.
    """
    question = state["question"]
    documents = state.get("documents", [])
    if documents:
        context = "\n".join([doc.page_content for doc in documents])
        prompt = f"Using the following context, answer the question:\n\nContext:\n{context}\n\nQuestion: {question}"
    else:
        prompt = f"Answer the question without additional context:\n\nQuestion: {question}"
    answer = llm.invoke(prompt)
    answer = answer.content
    return {**state, "answer": answer, "needs_retrieval": False}

In [ ]:
def should_retrieve(state: AgentState) -> str:
    """
    Decide whether to retrieve documents based on the current state.
    """
    if( state.get("needs_retrieval", True) or not state.get("documents")):
        return "retrieve"
    return "generate"

In [ ]:
# Build the State Graph
workflow = StateGraph(AgentState)
workflow.add_state("decide", decide_retrieval)
workflow.add_state("retrieve", retrieve_documents)
workflow.add_state("generate", generate_answer)


workflow.set_entry_point("decide")

workflow.add_conditional_edges(
    "decide",
    should_retrieve,
    {
        "retrieve": "retrieve", 
        "generate": "generate"
    }
)

workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()
app


In [ ]:
def ask_question(question: str) -> str:
    initial_state: AgentState = {
        "question": question,
        "documents": [],
        "answer": "",
        "needs_retrieval": True
    }
    final_state = app.run(initial_state)
    return final_state["answer"]